# 04 Evaluation

This notebook compares content-based, collaborative, and hybrid recommendation outputs using the reusable evaluator classes in `src/evaluation`.

The notebook is intentionally lightweight: metric logic stays in `src/`, while the notebook focuses on experiment setup, metric comparison, and interpretation.

## Setup

We add `src/` to the notebook path so the repository modules work cleanly in VS Code on Mac.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from evaluation.diversity_evaluator import DiversityEvaluator
from evaluation.ranking_evaluator import RankingEvaluator
from models.base_recommender import RecommendationResult


## Example Offline Evaluation Inputs

In a real offline experiment, these lists would come from held-out user interactions or playlist continuation targets. Here we use a small deterministic example so the notebook stays easy to explain in a portfolio walkthrough.

In [ ]:
# Ground-truth relevant tracks for one example user.
relevant_item_ids = {"track_1", "track_3", "track_5"}

# Example recommendation outputs from three model families.
content_recommendations = [
    RecommendationResult(item_id="track_1", score=0.98, source="content"),
    RecommendationResult(item_id="track_2", score=0.92, source="content"),
    RecommendationResult(item_id="track_4", score=0.88, source="content"),
    RecommendationResult(item_id="track_6", score=0.82, source="content"),
]
collaborative_recommendations = [
    RecommendationResult(item_id="track_3", score=0.97, source="collaborative"),
    RecommendationResult(item_id="track_5", score=0.90, source="collaborative"),
    RecommendationResult(item_id="track_2", score=0.85, source="collaborative"),
    RecommendationResult(item_id="track_7", score=0.80, source="collaborative"),
]
hybrid_recommendations = [
    RecommendationResult(item_id="track_3", score=0.99, source="hybrid"),
    RecommendationResult(item_id="track_1", score=0.95, source="hybrid"),
    RecommendationResult(item_id="track_5", score=0.91, source="hybrid"),
    RecommendationResult(item_id="track_8", score=0.76, source="hybrid"),
]

# Popularity values are treated as probability-like normalized values.
popularity_by_item_id = {
    "track_1": 0.70,
    "track_2": 0.85,
    "track_3": 0.50,
    "track_4": 0.40,
    "track_5": 0.35,
    "track_6": 0.20,
    "track_7": 0.25,
    "track_8": 0.15,
}

# Pairwise similarity is used to estimate intra-list diversity.
pairwise_similarity_by_item_ids = {
    ("track_1", "track_2"): 0.80,
    ("track_1", "track_4"): 0.65,
    ("track_1", "track_6"): 0.40,
    ("track_2", "track_4"): 0.75,
    ("track_2", "track_6"): 0.45,
    ("track_4", "track_6"): 0.55,
    ("track_3", "track_5"): 0.60,
    ("track_3", "track_2"): 0.50,
    ("track_3", "track_7"): 0.35,
    ("track_5", "track_2"): 0.42,
    ("track_5", "track_7"): 0.28,
    ("track_2", "track_7"): 0.30,
    ("track_3", "track_1"): 0.55,
    ("track_3", "track_8"): 0.22,
    ("track_1", "track_5"): 0.48,
    ("track_1", "track_8"): 0.20,
    ("track_5", "track_8"): 0.18,
}

catalog_item_ids = [f"track_{index}" for index in range(1, 11)]


## Instantiate Evaluators

We keep ranking metrics and beyond-accuracy metrics in separate evaluator classes because they answer different product questions.

In [ ]:
ranking_evaluator = RankingEvaluator()
diversity_evaluator = DiversityEvaluator()


## Compare Models With Ranking Metrics

These metrics tell us how well each recommender retrieves relevant tracks near the top of the list.

In [ ]:
model_outputs = {
    "content": content_recommendations,
    "collaborative": collaborative_recommendations,
    "hybrid": hybrid_recommendations,
}

ranking_rows = []
for model_name, recommendations in model_outputs.items():
    ranking_rows.append(
        {
            "model": model_name,
            "precision@3": ranking_evaluator.precision_at_k(recommendations, relevant_item_ids, 3),
            "recall@3": ranking_evaluator.recall_at_k(recommendations, relevant_item_ids, 3),
            "ndcg@3": ranking_evaluator.ndcg_at_k(recommendations, relevant_item_ids, 3),
            "map@3": ranking_evaluator.map_at_k(recommendations, relevant_item_ids, 3),
        }
    )

pd.DataFrame(ranking_rows).round(4)


## Compare Models With Beyond-Accuracy Metrics

These metrics help us reason about discovery quality, catalog health, and the risk of repetitive or popularity-heavy recommendation behavior.

In [ ]:
beyond_accuracy_rows = []
for model_name, recommendations in model_outputs.items():
    beyond_accuracy_rows.append(
        {
            "model": model_name,
            "diversity": diversity_evaluator.diversity(recommendations, pairwise_similarity_by_item_ids),
            "novelty": diversity_evaluator.novelty(recommendations, popularity_by_item_id),
            "coverage": diversity_evaluator.coverage([recommendations], catalog_item_ids),
            "popularity_bias": diversity_evaluator.popularity_bias(recommendations, popularity_by_item_id),
        }
    )

pd.DataFrame(beyond_accuracy_rows).round(4)


## Interpretation

In this toy example, the hybrid model should look strongest on ranking quality because it combines behavior and content evidence. At the same time, we still need beyond-accuracy metrics to check whether it is becoming too repetitive or too popularity-heavy.

That is the main evaluation lesson for music recommendation: a product can look accurate in offline ranking metrics while still feeling repetitive, safe, or difficult to discover new music through.